# SETUP

## LlamaIndex-Specific

In [1]:
import os
from transformers import set_seed

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
set_seed(42, deterministic=True)

import os
import pandas as pd

from llama_index.core import Document, Settings, VectorStoreIndex, PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.readers.database import DatabaseReader
from sqlalchemy import create_engine
from tqdm import tqdm

/home/luthfi/miniconda3/envs/rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
query_wrapper_prompt = PromptTemplate(
    "[INST] {query_str} [/INST]"
)  # Taken from Mistral-7B-Instruct-v0.3's chat template
Settings.embed_model = HuggingFaceEmbedding(
    model_name="../../../src/processor/model/weight/bge-base"
)
Settings.llm = HuggingFaceLLM(
    context_window=32768,
    max_new_tokens=1000,
    query_wrapper_prompt=query_wrapper_prompt,
    generate_kwargs={"do_sample": False, "pad_token_id": 2},
    model_kwargs={
        "torch_dtype": torch.bfloat16,
    },
    tokenizer_name="../../../src/processor/model/weight/mistral-7b",
    model_name="../../../src/processor/model/weight/mistral-7b",
    device_map="auto",
    tokenizer_kwargs={"max_length": 32768},
)

Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.30s/it]


In [3]:
# Adjust names
DATASET = "environment"
DATA_SRC_DIR = "../../../data_src"

In [4]:
def get_documents(path: str, duckdb_filename: str):
    engine = create_engine(f"duckdb:///index/{duckdb_filename}.duckdb")
    contexts = pd.read_csv(f"{path}/metadata.csv")
    db_reader = DatabaseReader(engine)
    final_documents = []

    for table in tqdm(
        sorted([table[:-4] for table in os.listdir(f"{path}/dataset")]),
        desc="Process tables",
    ):
        # Inserting contents
        documents = db_reader.load_data(f"select * from '{table}'")
        for i in range(len(documents)):
            documents[i].id_ = f"{table}.csv_part_{i}"
            documents[i].text = documents[i].id_ + f": {documents[i].text}"
            final_documents.append(documents[i])

        # Inserting contexts
        specific_contexts = [
            context["description"]
            for _, context in contexts.iterrows()
            if context["table_name"] == f"{table}.csv"
        ]
        base_id = len(documents)
        for j in range(len(specific_contexts)):
            document = Document(
                text=f"{table}.csv_part_{base_id+j}: " + specific_contexts[j],
                doc_id=f"{table}.csv_part_{base_id+j}",
            )
            final_documents.append(document)
    return final_documents
documents = get_documents(f"{DATA_SRC_DIR}/{DATASET}", DATASET)

Process tables: 100%|██████████| 36/36 [00:12<00:00,  2.87it/s]


In [5]:
index = VectorStoreIndex.from_documents(documents, show_progress=True)

Generating embeddings: 100%|██████████| 1458/1458 [00:03<00:00, 384.32it/s]


In [6]:
query_engine = index.as_query_engine(similarity_top_k=10)

## Benchmark-specific

In [7]:
import json
import sys
from dotenv import load_dotenv

load_dotenv("../../../.env")
sys.path.append("../../../src")

from processor.model.interface.impl.gpt import GPT

In [8]:
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATASET}.jsonl")
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [9]:
gpt = GPT("gpt-4o-mini")

In [10]:
def get_format_to_gpt(output: str):
    return f"SYSTEM OUTPUT: ```{output}```"

def get_initial_prompt_to_chatgpt(domain: str, question: str):
    domain_expert_desc = f"a {domain} domain expert"
    if domain == "archeology":
        domain_expert_desc = "a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration"
    return f"""You are simulating {domain_expert_desc}, who is interacting with a data assistant system to explore insights from an enterprise dataset.

In this scenario, the system already has access to internal environment-related datasets. You (the simulated user) are already somewhat familiar with the topics of the dataset, as it is commonly used in your team or organization. You are not uploading a new dataset or asking about the existence of some dataset. Your task is to gradually explore or refine your information need about some aspect of the data. You do not begin with a precise question; rather, your curiosity evolves based on system responses and your domain expertise.

Here is a possible eventual goal (you do not know this yet, but may arrive at it through exploration):

{question}

Your behavior should reflect the following:
- You are familiar with the domain.
- You explore and refine your question step-by-step depending on the system's ability to surface relevant information.
- You may be vague or even explore tangents, just as a curious analyst would when exploring data without a clear goal.
- You will only arrive at the specific question above if the system's output correctly leads you there.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {INITIAL_PROMPT}"""

# EXPERIMENT

In [38]:
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption
index = 5

## Answer Generation

In [39]:
INITIAL_PROMPT = benchmark[index]["interactive_initial_prompt"]
print(f"Original (direct) question: {benchmark[index]["original_direct_question"]}")
ITERATION_LIMIT = 15
gpt_init_prompt = get_initial_prompt_to_chatgpt(
    DATASET, benchmark[index]["original_direct_question"]
)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = INITIAL_PROMPT
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")
for iteration in tqdm(range(ITERATION_LIMIT)):
    system_output = query_engine.query(curr_user_prompt).response
    print(f"===> SYSTEM OUTPUT: {system_output}")
    format_to_gpt = get_format_to_gpt(system_output)
    if iteration == 0:
        gpt_messages[0]['content'] += f"\n{format_to_gpt}"
    else:
        gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
    updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
    gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
    if updated_user_prompt.startswith("YOU:"):
        updated_user_prompt = updated_user_prompt[4:]
        updated_user_prompt = updated_user_prompt.strip()
    curr_user_prompt = updated_user_prompt
    print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")
write_jsonl(f"benchmark_data/{DATASET}_{index+1}.jsonl", gpt_messages, True)

Original (direct) question: What is the average historical exceedance rate (to 2 decimal places) for marine beaches from 2002 to 2023 (inclusive)?
=> CURRENT USER PROMPT: I'm trying to get a better understanding of how marine beach water quality has been monitored over time. Could you help me explore trends or patterns in the available data across the past couple of decades?


  0%|          | 0/15 [00:00<?, ?it/s]

===> SYSTEM OUTPUT: To help you explore trends or patterns in the marine beach water quality data across the past couple of decades, I would recommend analyzing the available datasets for Boston-area marine beaches. Here's a list of datasets that contain information about marine beaches in Boston:

1. M Street Beach, South Boston: Bacterial Water Quality (m_street_beach_datasheet.csv_part_1117)
2. Tenean Beach, Dorchester: Bacterial Water Quality (tenean_beach_datasheet.csv_part_1880)
3. City Point Beach, South Boston: Bacterial Water Quality (city_point_beach_datasheet.csv_part_1025)
4. Constitution Beach, East Boston: Bacterial Water Quality (constitution_beach_datasheet.csv_part_1882)
5. Malibu Beach, Dorchester: Bacterial Water Quality (malibu_beach_datasheet.csv_part_1158)
6. Wollaston Beach, Quincy: Bacterial Water Quality (wollaston_beach_datasheet.csv_part_1904)
7. Pleasure Bay Beach, South Boston: Bacterial Water Quality (pleasure_bay_and_castle_island_beach_datasheet.csv_part

  7%|▋         | 1/15 [00:55<13:02, 55.90s/it]

=> CURRENT USER PROMPT: Thanks for the list of datasets! I see that there are several marine beaches in Boston with bacterial water quality data. I'm particularly interested in understanding how the water quality has changed over time. Could you provide me with a summary of the historical data available for these beaches? For instance, what years do the datasets cover, and are there any notable trends in the bacterial levels?
===> SYSTEM OUTPUT: Based on the provided context, I can inform you that the datasets cover different years for each beach. Here's a summary of the years the datasets cover for the marine beaches in Boston:

1. M Street Beach, South Boston: Data available up to 2011 (m_street_beach_datasheet.csv_part_1117)
2. City Point Beach, South Boston: Data available up to 2010 (city_point_beach_datasheet.csv_part_1025)
3. Pleasure Bay Beach, South Boston: Data available up to 2008 (pleasure_bay_and_castle_island_beach_datasheet.csv_part_860)
4. Constitution Beach, East Bosto

...

=> CURRENT USER PROMPT: Thank you for the reminder about M Street Beach! I’ll focus on that along with the other beaches I have data for. I appreciate your willingness to assist, and I’ll definitely reach out if I have any questions or need further guidance. Looking forward to getting started on this analysis!
===> SYSTEM OUTPUT: You're welcome! I'm glad to help you with your analysis on M Street Beach. If you have data for other beaches as well, feel free to include them in your study. I'm here to assist you with any questions or guidance you might need. Let's get started on this interesting analysis!


 93%|█████████▎| 14/15 [08:47<00:27, 27.51s/it]

=> CURRENT USER PROMPT: Absolutely! I plan to include data from all the marine beaches I have access to, not just M Street Beach. This will give me a broader perspective on water quality trends across the area. 

I’m excited to get started and will reach out if I need any assistance or have questions along the way. Thank you for your support!
===> SYSTEM OUTPUT: That sounds like a great approach! By analyzing data from multiple marine beaches in the Boston area, you'll indeed gain a more comprehensive understanding of water quality trends.

To help you get started, here's a brief summary of the data you have for M Street Beach:

1. M Street Beach is a marine beach located in Boston, Massachusetts.
2. It has been tested for Enterococci levels in 2021, 2022, and 2023.
3. The Enterococci levels have ranged from 1 to 41, with some instances where the levels were below the indicator level of 1.0 (which did not result in a violation).
4. The beach has experienced varying amounts of precipita

100%|██████████| 15/15 [09:23<00:00, 37.59s/it]

=> CURRENT USER PROMPT: Thank you for the detailed summary of the data for M Street Beach! It’s helpful to know the range of Enterococci levels and the precipitation data. I’ll definitely consider those factors as I analyze the trends.

I’ll also keep in mind the potential influences of nearby pollution sources and historical patterns as I gather data from the other beaches. If I have any questions or need further clarification, I’ll be sure to reach out. I appreciate your support and guidance!


## Convergence Checking

In [40]:
benchmark_data = read_jsonl(f"benchmark_data/{DATASET}_{index+1}.jsonl")
filtered_benchmark_data = [i for i in benchmark_data if i['role'] == 'assistant']
filtered_benchmark_data[0]
actual_benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATASET}.jsonl")[index]
actual_benchmark["original_direct_question"]
def get_eval_prompt(hidden, bench_data):
    return (f"""You are evaluating whether a simulated domain expert has successfully converged on a target information need.

Below is the target (hidden) goal question:
---
{hidden}
---

Below is the most recent query (or set of queries) made by the simulated expert:
---
{bench_data}
---

Does the user's most recent query express the same information need as the hidden goal, either literally or in semantically equivalent terms?

Respond with only one of the following labels:
- CONVERGED (if CONVERGED, tell me exactly in which interaction convergence occurs)
- NOT CONVERGED (explain a little why)""")

In [41]:
eval_message = [
    LLMMessage(
        role=Role.SYSTEM.value,
        content=get_eval_prompt(
            actual_benchmark["original_direct_question"],
            filtered_benchmark_data
        )
    )
]
gpt.chat(eval_message)

'NOT CONVERGED\n\nThe user has not directly asked for the average historical exceedance rate for marine beaches from 2002 to 2023. Instead, the focus has been on analyzing the available data, understanding bacterial levels, and calculating exceedance rates based on the limited data available. While they plan to determine exceedance rates, they have not yet engaged in a query that explicitly states the intent to find the average historical exceedance rate for the specified years. The convergence does not occur in the interactions presented.'